# LLMs in Finance
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Use an LLM to extract structured information** from earnings transcripts and Fed statements
2. **Build a sentiment / tone score** from a 10-K or call transcript
3. **Recognize look-ahead bias** in LLM-based features (training data ends after your "investment date")
4. **Compare LLM-extracted signals** to traditional NLP (dictionary methods)
5. **Audit LLM output** — hallucination, sycophancy, model version sensitivity

## 📋 TOC
1. [Setup](#setup)  2. [Why LLMs for Finance](#why)
3. [Pitfall Checklist](#pitfalls)  4. [Earnings Call: Tone Extraction](#tone)
5. [FOMC: Hawkish / Dovish](#fomc)  6. [The Look-Ahead Trap](#lookahead)
7. [🎯 Challenge: Score Three Transcripts](#challenge)
8. [Submission](#submit)  9. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## Why LLMs for Finance <a id="why"></a>

Half of financial information is **text** — earnings calls, 10-Ks, press releases,
Fed statements, news articles. Pre-LLM, extracting structured features from
text required:
- Dictionary methods (Loughran-McDonald sentiment lists)
- Regex parsing
- Naïve Bayes / SVM classifiers

LLMs make this dramatically easier:
- Pass the text, ask a question, get a structured answer
- No training data required for most tasks
- Zero-shot performance on previously hard tasks (entity extraction, summary, tone)

**The catch:** look-ahead bias, hallucination, and prompt sensitivity. We
cover these next.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Training data lookahead** | LLM was trained on 2024 data; you ask it about 2018 events with retrospective knowledge | Always ask "what was known at the time" or use a model frozen before your date |
| 2 | **Entity confusion** | LLM confuses parent company with subsidiary, or two firms with similar names | Specify CIK / ticker; verify the entity in output |
| 3 | **Hallucination** | LLM makes up a CEO name or a number that wasn't in the text | Ground every fact in the source text; reject anything you can't trace back |
| 4 | **Prompt sensitivity** | Rewording the question gives different answers | Run the same prompt 5x; check for consistency |
| 5 | **Sycophancy** | LLM tells you what you want to hear ("Yes, this is bullish") | Always use neutrally-worded prompts; ask for evidence |
| 6 | **Cost at scale** | $0.01/call × 10000 firms × 4 quarters × 30 years = $$$ | Use smaller / faster models for batch; reserve big models for nuance |

---
## Earnings Call: Tone Extraction <a id="tone"></a>

The classic financial-NLP task: given an earnings call transcript, extract:
- **Tone** (positive / negative / neutral)
- **Forward-looking statements** (guidance, expectations)
- **Surprises** (what's different from last quarter)

Pre-LLM: Loughran-McDonald dictionary count of positive/negative words.
With LLM: pass the transcript and ask.

> **🤖 AI prompt:**
>
> *"Below is a transcript of an earnings call for [TICKER] on [DATE]. Identify:
> (1) the 3 most positive forward-looking statements, (2) the 3 most negative
> or cautious statements, (3) one statement that surprised relative to prior guidance.
> Quote the exact text for each. Do not infer beyond what's stated."*

---
## FOMC: Hawkish / Dovish <a id="fomc"></a>

Same workflow, different domain. Fed statements are short (1-2 pages) but
each word matters — markets move 5-10 bps off a single word change.

**Hawkish** = more inflation-focused, more likely to raise rates.
**Dovish** = more growth-focused, more likely to cut rates.

LLM can read both the latest statement and the prior one, then highlight
the **differences** in tone — exactly what bond traders do manually.

---
## The Look-Ahead Trap <a id="lookahead"></a>

This is the **most important** pitfall for LLMs in finance:

If you ask GPT-4 about Apple's earnings in Q3 2018, **GPT-4 was trained on
data through 2023**. It knows what happened next.

If you use its analysis to backtest a Q3 2018 trading strategy, you have a
massive lookahead bias.

**Mitigations:**
- Use models with a known cutoff before your data
- Explicitly prompt: "Pretend it's [DATE]. Don't use information from after this date."
- For research, use frozen historical models (some providers offer dated snapshots)

> **⚠️ This is THE bias to watch for**
>
> Every academic paper using LLMs on historical data has to defend against
> this. Many fail to.

---
## 🎯 Challenge: Score Three Transcripts <a id="challenge"></a>

> **Setup.** You have three fake snippets — score each one for tone.
> (In a real workflow you'd pass each to an LLM. For class purposes we provide
> ground-truth scores and have you do the bookkeeping.)

### Q1 — Score each snippet

The snippets are below. Assign each a tone score from -1 (very negative) to +1 (very positive).

> **📌 Required:**
> ```python
> # Snippet 1: "We exceeded guidance by 20%. Customer demand is the strongest in our history."
> # Snippet 2: "We faced material headwinds. Some segments declined double digits. We're rebuilding."
> # Snippet 3: "Revenue was in line. We expect similar growth next quarter."
>
> tone_snippet_1 = ____   # in [-1, 1]
> tone_snippet_2 = ____
> tone_snippet_3 = ____
> ```

In [ ]:
tone_snippet_1 = ____
tone_snippet_2 = ____
tone_snippet_3 = ____
print(f"Snippet 1 tone: {tone_snippet_1:+.2f}")
print(f"Snippet 2 tone: {tone_snippet_2:+.2f}")
print(f"Snippet 3 tone: {tone_snippet_3:+.2f}")

### Q2 — Composite tone

If you bought all 3 stocks equally, what's the average tone you'd hold?

> **📌 Required:**
> ```python
> avg_tone = ____   # (s1 + s2 + s3) / 3
> ```

In [ ]:
avg_tone = ____
print(f"Average tone: {avg_tone:+.2f}")

### Q3 — Lookahead check
Suppose your LLM was trained on data through 2024-12. You're applying it
to score 2020 earnings calls. Is there look-ahead risk?

> **📌 Required:**
> ```python
> lookahead_risk = ____    # 1.0 if yes, 0.0 if no
> ```

In [ ]:
lookahead_risk = ____
print(f"Look-ahead risk present? {bool(lookahead_risk)}")

### Q4 — Memo

Max 5 sentences. State (i) the average tone, (ii) the look-ahead concern,
(iii) one mitigation you'd use in a real backtest.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["tone_snippet_1", "tone_snippet_2", "tone_snippet_3", "avg_tone", "lookahead_risk", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "LLMs_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **LLMs make text-feature extraction dramatically easier** — earnings calls, Fed statements, 10-Ks.
2. **Look-ahead bias is THE big risk.** The model knows what happened after your "investment date."
3. **Audit every fact** the LLM gives you against the source text.
4. **Prompt sensitivity is real** — wording matters; consistency-test by re-running.
5. **You direct the LLM. You verify the output. You're responsible for the answer.**